In [ ]:
import json
import os
from pathlib import Path
from pprint import pprint
from typing import Optional, Any
from collections import Counter

import torch
import torch.nn as nn
import numpy as np
import wandb
from torch import Tensor
import torch.nn as nn

from datasets import load_from_disk
from transformers import AutoModelForTokenClassification, AutoTokenizer,\
DataCollatorForTokenClassification, TrainingArguments, Trainer, EvalPrediction
import evaluate

from colorama import Fore, Style

In [ ]:
from transformers.models.deberta_v2.tokenization_deberta_v2 import DebertaV2Tokenizer
from transformers.models.deberta_v2.modeling_deberta_v2 import DebertaV2ForTokenClassification
from datasets import DatasetDict

In [ ]:
# remove transformers unneeded diagnostic messages
from transformers.utils import logging
logging.set_verbosity_error()

In [ ]:
MODEL_PATH = "microsoft/deberta-v3-xsmall"

In [ ]:
DATA_DIR = Path(os.getcwd()).parent / "data"
DATA_DIR.mkdir(exist_ok=True)

LABEL_INFO_PATH = DATA_DIR/"label_info.json"
DATASET_PATH = DATA_DIR/"cleaned_ai4privacy_300k_pii"

MODEL_DIR = Path(os.getcwd()).parent / "models"
MODEL_DIR.mkdir(exist_ok=True)

In [ ]:

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR),
    learning_rate=1e-3,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    warmup_ratio=0.1,
    bf16=True,
    fp16=False,
    tf32=False,
    max_grad_norm=1.0,
    disable_tqdm=False
)

def update_args():
    # updated training args
    training_args.learning_rate = 2e-5 # lower learning rate
    training_args.num_train_epochs = 5
    # lower batch size now that gradient tree is bigger
    training_args.per_device_train_batch_size = 32
    training_args.gradient_accumulation_steps = 1

In [ ]:
dataset: DatasetDict = load_from_disk(DATASET_PATH) # type:ignore

In [ ]:
# label info
with open(LABEL_INFO_PATH) as f:
    label_info: dict = json.load(f)
label2id: dict[str, int] = label_info["label2id"]
id2label = label_info["id2label"]

In [ ]:
tokenizer: DebertaV2Tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model: DebertaV2ForTokenClassification = AutoModelForTokenClassification.from_pretrained(
    MODEL_PATH, num_labels=len(id2label), id2label=id2label, label2id=label2id)

assert tokenizer.is_fast

In [ ]:
for name, param in model.named_parameters():
    if torch.isnan(param).any():
        print(f"NaN in: {name}")

In [ ]:
for param in model.classifier.named_parameters():
    print(param)

In [ ]:
Counter([param.dtype for param in model.parameters()])

In [ ]:
for name, module in model.named_modules():
    module.to(dtype=torch.float32) # type:ignore

In [ ]:
# make model dtype float32 to prevent gradients exploding
Counter([param.dtype for param in model.parameters()])

In [ ]:
def tokenize_and_align_labels_single(example):
    """tokenize and align labels for deberta tokenizer.
    since deberta tokenizer will mark space before as part of the next word,
    masks that start the first letter (not white-space) needs this tokenizer.
    it will first get the word_ids that fall in the mask, then label for
    seqeval

    Args:
        example (_type_): _description_

    Returns:
        _type_: _description_
    """
    encoding = tokenizer(
        example["source_text"],
        return_offsets_mapping=True,
        add_special_tokens=True,
        truncation=True,
        max_length=512
    )
    
    word_ids= encoding.word_ids()
    offsets= encoding["offset_mapping"]
    
    # map offsets to their word_id - for sentence-piece tokenization which 
    # tokenizes the space before the word as part of that word
    word_span: dict[int, tuple[int, int]] = {}
    for offset, word_id in zip(offsets, word_ids):
        if word_id is None:
            continue
        if word_id not in word_span:
            word_span[word_id] = offset
        else:
            prev_start, prev_end = word_span[word_id]
            word_span[word_id] = (
                min(prev_start, offset[0]),
                max(prev_end, offset[1])
            )
    
    # map every word_id to its BI tag
    masks = example["privacy_mask"]
    word_to_ent: dict[int, str] = {}
    for mask in masks:
        words_in_mask = sorted(
            word_id
            for word_id, (w_start, w_end) in word_span.items()
            if w_start < mask["end"] and w_end > mask["start"] 
        )

        for word_id in words_in_mask:
            word_to_ent[word_id] = mask["label"]
    
    # add labels and bio tags
    labels: list[int] = []
    bio_tags: list[Optional[str]] = []
    prev_word_id = None
    prev_ent = None
    for word_id in word_ids:
        if word_id is None:
            labels.append(-100)
            bio_tags.append(None)
            
        elif word_id not in word_to_ent:
            labels.append(label2id["O"])
            bio_tags.append("O")
        else: 
            entity = word_to_ent[word_id] 
            
            if word_id != prev_word_id:
                tag = f"I-{entity}" if prev_ent == entity else f"B-{entity}"
                labels.append(label2id[tag])
            else:
                tag = f"I-{entity}"
                labels.append(-100)
                
            bio_tags.append(tag)
            prev_ent = entity
        prev_word_id = word_id
    
    encoding["ner_tags"] = bio_tags
    encoding["labels"] = labels
    return encoding

In [ ]:
dataset = dataset.map(tokenize_and_align_labels_single, batched=False)

In [ ]:
example = dataset["train"][0]
example_encodings = tokenizer(
    example["source_text"],
    return_offsets_mapping=True,
    add_special_tokens=True,
)
line1 = ""
line2 = ""
line3 = ""
line4 = ""
for word, tag, label, word_id in zip(example_encodings.tokens(), example["ner_tags"], example["labels"], example_encodings.word_ids()):
    tag = str(tag)
    word_id = str(word_id)
    label = str(label)
    max_length = max(len(word), len(tag), len(word_id), len(label))
    line1 += word + " " * (max_length - len(word) + 1)
    line2 += tag + " " * (max_length - len(tag) + 1)
    line3 += label + " " * (max_length - len(label) + 1)
    line4 += word_id + " " * (max_length - len(word_id) + 1)
    
print(line1)
print(line2)
print(line3)
print(line4)

In [ ]:
# some tokens include whitespace
print(example["source_text"][310:313].replace(" ", "_"))

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

seqeval = evaluate.load("seqeval")

In [ ]:
def compute_metrics(p: EvalPrediction) -> dict[str, float]:
    predictions, label_ids = p.predictions, p.label_ids
    predictions = np.argmax(predictions, axis=-1)
    
    true_preds = [
        [id2label[str(pred)] for pred, tgt_id in zip(preds_row, labels_row) if tgt_id != -100]
        for preds_row, labels_row in zip(predictions, label_ids)
    ]
    
    true_labels =[
        [id2label[str(tgt_id)] for tgt_id in labels_row if tgt_id != -100]
        for labels_row in label_ids
    ]
    
    results = seqeval.compute(predictions=true_preds, references=true_labels)
    
    flat = {}
    if results is not None:
        flat.update({
            "precision": results["overall_precision"],
            "recall": results["overall_recall"],
            "f1": results["overall_f1"],
            "accuracy": results["overall_accuracy"],
        })
        
        for entity, scores in results.items():
            if isinstance(scores, dict): # filter scores for individual labels
                flat[f"{entity}_f1"] = scores["f1"]
                flat[f"{entity}_support"] = scores["number"]
    
    return flat

In [ ]:
class WeightedTokenClassificationTrainer(Trainer):
    def __init__(self, *args, class_weights: Optional[torch.Tensor]=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        
    def compute_loss(
        self, 
        model: nn.Module, 
        inputs: dict[str, Tensor | Any], 
        return_outputs: bool = False, 
        num_items_in_batch: Tensor | int | None = None
    ) -> Tensor | tuple[Tensor, Any]:
        # get target labels from the inputs
        labels = inputs.get("labels")
        # forward pass
        outputs = model(**inputs)
        
        if labels is None:
            # if no labels during training raise error
            if model.training:
                raise ValueError(
                    "Labels are required during training for WeightedTokenClassificationTrainer."
                )
            # silent fall back for inference (when labels aren't provided)
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs.loss
            return (loss, outputs) if return_outputs else loss
        
        logits = outputs["logits"] if isinstance(outputs, dict) else outputs.logits
        
        # get class weights and move to device
        weight = None
        if self.class_weights is not None:
            weight = self.class_weights.to(logits.device)
            
        # def loss function with label weights and ignore -100 in labels    
        loss_fct = torch.nn.CrossEntropyLoss(weight=weight, ignore_index=-100)
        # get loss by comparing predictions to labels
        loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
        
        # for gradient accumulation
        if num_items_in_batch is not None:
            loss = loss/num_items_in_batch
            
        return (loss, outputs) if return_outputs else loss


In [ ]:
from transformers import TrainerCallback

class DetailedProgressCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and state.is_local_process_zero:
            step = state.global_step
            total = state.max_steps
            loss = logs.get("loss")
            lr = logs.get("learning_rate")
            eval_loss = logs.get("eval_loss")

            if loss is not None and lr is not None:
                # Training step log
                print(f"  Step {step}/{total} | loss: {loss:.4f} | lr: {lr:.2e}")
            elif eval_loss is not None:
                # Eval log
                f1 = logs.get("eval_f1", "—")
                f1_str = f"{f1:.4f}" if isinstance(f1, float) else f1
                print(f"  Eval step {step} | eval_loss: {eval_loss:.4f} | f1: {f1_str}")

In [ ]:
labels_counter = Counter()
for labels in dataset["train"]["labels"]:
    labels_counter.update(labels)

# remove -100 and make tensor
labels_count_tensor = torch.tensor(
    [labels_counter[i] for i in id2label.keys()],
    dtype=torch.float
)
    
weights = 1.0 / labels_count_tensor.clamp(min=1.0)
weights = weights / weights.sum() # normalize
weights

In [ ]:
# for name, param in model.named_parameters():
#     param.requires_grad = False if not name.startswith("classifier") else True
#     if param.requires_grad:
#         print(f"{name} requires grad")

for param in model.base_model.parameters():
    param.requires_grad = False
    
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name} requires grad")

trainer = WeightedTokenClassificationTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[DetailedProgressCallback()],
    class_weights=weights
)
trainer_output = trainer.train()

In [ ]:
for name, param in model.named_parameters():
    if torch.isnan(param).any():
        print(f"NaN in: {name}")

In [ ]:
import pandas as pd
df = pd.DataFrame( trainer_output)

In [ ]:
for item in trainer_output:
    print(item)

In [ ]:
# Unfreeze the backbone
for param in model.base_model.parameters():
    param.requires_grad = True

update_args()

trainer = WeightedTokenClassificationTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[DetailedProgressCallback()],
    class_weights=weights
)

trainer.train()

In [ ]:
for name, param in model.named_parameters():
    if torch.isnan(param).any():
        print(f"NaN in: {name}")

In [ ]:
pred_output = trainer.predict(
    test_dataset=dataset["test"]
)

In [ ]:
pprint(pred_output.metrics)

In [ ]:
min(pred_output.metrics.items())

In [ ]:
f1s = []
for metric, value in pred_output.metrics.items():
    if metric.endswith("f1"):
        f1s.append((metric, value))
        
sorted_f1s = sorted(f1s, key=lambda x: x[1], reverse=True)

print("worst f1s:")
for metric, value in reversed(sorted_f1s[-7:]):
    print(f"{metric:<20}= {value:.3f}")

In [ ]:
Counter([param.dtype for param in model.parameters()])